# Model 06 — Linear Support Vector Machine

## BRFSS 2025 Diabetes Classification

**Purpose:** execute and freeze the sixth classifier without rerunning Models 01–05.

### Literature provenance

**BRFSS application — Xie et al. (2019), Preventing Chronic Disease / CDC**, DOI `10.5888/pcd16.190109`.

That study evaluated three SVM kernels on BRFSS 2014: linear, RBF, and polynomial. Among their SVM variants, the linear SVM had the highest reported AUC.

**BRFSS application — Muhammad et al. (2025)**, DOI `10.1177/21501319251400546`.

That study included Support Vector Machine among seven BRFSS 2023 diabetes classifiers using stratified 5-fold CV.

**Architecture source — Cortes & Vapnik (1995), Support-Vector Networks**, DOI `10.1007/BF00994018`.

Model 06 intentionally uses a **linear SVM** rather than an RBF kernel because the transformed BRFSS dataset is large and sparse, and linear SVM has direct BRFSS precedent.


## 1. Frozen Study Protocol

Inherited unchanged from Part II:

- `DIABETE4=1` vs `DIABETE4=3`;
- frozen 20 predictors;
- 80/20 stratified split with `random_state=42`;
- categorical missing category + one-hot encoding;
- continuous median imputation + standardization;
- Stratified 5-Fold CV;
- no SMOTE or class reweighting;
- untouched internal test set.

### Important metric difference

`LinearSVC` does not produce calibrated probabilities.

Therefore:
- Accuracy, Precision, Recall, Specificity, and F1 use predicted labels.
- ROC-AUC and PR-AUC use the continuous `decision_function` margin.
- The class boundary is **decision score = 0**.
- **Brier score is not reported** because applying it to uncalibrated margins would be invalid.

Probability calibration, if desired, must be a separate experiment.


In [ ]:
from pathlib import Path
from math import sqrt
import json
import platform
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "DIABETE4"

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

xpt_candidates = glob.glob("/kaggle/input/**/LLCP2025.XPT", recursive=True)
if len(xpt_candidates) != 1:
    raise RuntimeError(f"Expected exactly one LLCP2025.XPT input, found: {xpt_candidates}")
xpt_path = Path(xpt_candidates[0])

print("Input:", xpt_path)
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)

    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)

    return out[PRIMARY_FEATURES]

target_mask = df[TARGET].isin([1.0, 3.0])
X = recode_brfss_2025(df.loc[target_mask, PRIMARY_FEATURES].copy())
y = (df.loc[target_mask, TARGET] == 1.0).astype("int8").rename("diabetes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

assert len(X) == 342_539
assert len(X_train) == 274_031
assert len(X_test) == 68_508
assert int(y.sum()) == 51_827

print("Cohort:", len(X))
print("Train:", len(X_train), "Test:", len(X_test))


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

specificity_scorer = make_scorer(recall_score, pos_label=0)
precision_scorer = make_scorer(precision_score, zero_division=0)
scoring = {
    "accuracy": "accuracy",
    "precision": precision_scorer,
    "recall": "recall",
    "specificity": specificity_scorer,
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}


## 2. Linear SVM Architecture

Support Vector Machines seek a separating hyperplane with a large margin between classes, while soft-margin regularization allows classification violations for non-separable data.

The fixed large-sample linear baseline is:

```python
LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual="auto",
    tol=1e-4,
    C=1.0,
    fit_intercept=True,
    class_weight=None,
    random_state=42,
    max_iter=10000,
)
```

### Provenance classification

**Literature-supported**
- SVM family and maximum-margin architecture;
- direct linear SVM use on BRFSS diabetes in P02;
- SVM comparator role in P03.

**Study-specific fixed choices**
- `LinearSVC` implementation;
- L2 + squared-hinge objective;
- `C=1.0`;
- automatic primal/dual selection;
- no class weighting;
- no SMOTE;
- zero-margin class threshold.

No RBF/poly kernel or calibration is added to this primary baseline.


In [ ]:
MODEL_06_DIR = Path("/kaggle/working/model_06_linear_svm")
MODEL_06_DIR.mkdir(parents=True, exist_ok=True)

linear_svm = LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual="auto",
    tol=1e-4,
    C=1.0,
    fit_intercept=True,
    class_weight=None,
    random_state=RANDOM_STATE,
    max_iter=10000,
)

svm_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", linear_svm),
])

config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "LinearSVC",
    "architecture": "soft-margin linear support vector classifier",
    "penalty": "l2",
    "loss": "squared_hinge",
    "dual": "auto",
    "tol": 1e-4,
    "C": 1.0,
    "fit_intercept": True,
    "class_weight": None,
    "random_state": RANDOM_STATE,
    "max_iter": 10000,
    "decision_threshold": 0.0,
    "probability_calibration": None,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_06_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


## 3. Five-Fold Cross-Validation

The full preprocessing + Linear SVM pipeline is fitted independently inside every training fold.

ROC-AUC and PR-AUC are computed from the signed decision margin rather than probabilities.


In [ ]:
svm_cv_raw = cross_validate(
    svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(svm_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(svm_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(svm_cv_raw["score_time"][fold])
    fold_rows.append(row)

svm_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = svm_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

svm_cv_summary = pd.DataFrame(summary_rows)
display(svm_cv_folds)
display(svm_cv_summary)

svm_cv_folds.to_csv(MODEL_06_DIR / "cv_folds.csv", index=False)
svm_cv_summary.to_csv(MODEL_06_DIR / "cv_summary.csv", index=False)


## 4. Fit Frozen Linear SVM and Evaluate the Untouched Test Set

The fixed model is fit once on the full training partition.

Class labels use the natural SVM boundary at decision margin zero. Ranking metrics use the continuous decision margin.


In [ ]:
svm_pipeline.fit(X_train, y_train)

y_test_score_svm = svm_pipeline.decision_function(X_test)
y_test_pred_svm = svm_pipeline.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_svm, labels=[0, 1]).ravel()

test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_svm)),
    "precision": float(precision_score(y_test, y_test_pred_svm, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_svm)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_svm)),
    "roc_auc": float(roc_auc_score(y_test, y_test_score_svm)),
    "pr_auc": float(average_precision_score(y_test, y_test_score_svm)),
    "brier_score": None,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "decision_threshold": 0.0,
}

model = svm_pipeline.named_steps["model"]
test_metrics["n_iter"] = int(np.max(np.atleast_1d(model.n_iter_)))

print(json.dumps(test_metrics, indent=2))

with open(MODEL_06_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)


## 5. Stratified Bootstrap Confidence Intervals

The same 200-replicate class-stratified bootstrap is used for all valid Linear SVM metrics.

Brier score is omitted because the SVM margins are not calibrated probabilities.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251008

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    score_b = y_test_score_svm[sample_idx]
    pred_b = (score_b > 0.0).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()
    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, score_b),
        "pr_auc": average_precision_score(y_b, score_b),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc"]:
    values = bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

test_summary = pd.DataFrame(test_summary_rows)
display(test_summary)

bootstrap_df.to_csv(MODEL_06_DIR / "bootstrap_metrics.csv", index=False)
test_summary.to_csv(MODEL_06_DIR / "test_summary.csv", index=False)


## 6. Diagnostics and Linear Coefficients

ROC and precision-recall curves use the SVM decision score.

The transformed-feature coefficients describe the fitted hyperplane. Their magnitudes are model diagnostics, not causal effects.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_score_svm)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_score_svm)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Linear SVM (AUC={test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 06 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Linear SVM (AP={test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 06 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 06 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = svm_pipeline.named_steps["preprocess"].get_feature_names_out()
coef = svm_pipeline.named_steps["model"].coef_.ravel()
coef_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "coefficient": coef,
    "abs_coefficient": np.abs(coef),
}).sort_values("abs_coefficient", ascending=False)

display(coef_table.head(20))
coef_table.to_csv(MODEL_06_DIR / "linear_coefficients.csv", index=False)


## 7. Freeze Rule

After verified execution, Model 06 is frozen as:

> **L2 Linear SVM with squared-hinge loss, C=1.0, automatic primal/dual selection, no class weighting, no resampling, zero-margin decision boundary.**

No post-test change to `C`, loss, class weighting, calibration, decision threshold, preprocessing, or feature set is allowed.

RBF/poly SVMs or probability-calibrated SVMs must be separate experiments.
